# Fine-tune Depth Anything V2 Metric Small on NYU Depth V2 (1/7)

First notebook of the **fine-tuning series** (one model at a time). This one fine-tunes **Depth Anything V2 Metric Indoor Small** — the same backbone used by the SAM2 segmentation+depth fusion in the comparison notebook — on the NYU Depth V2 **train** split, then evaluates it with the *exact same protocol* as `tsk3succeed.ipynb`:

- same validation samples (`sayakpaul/nyu_depth_v2`, first 100 of the validation split)
- same 7 metrics (AbsRel / SqRel / RMSE / RMSE-log / delta1-3), same valid-pixel mask, same `max_depth = 10`
- same `results/<slug>.json` format, so the fine-tuned model (`da2-metric-small-ft`) joins `model_metrics.csv` and every comparison chart automatically

**Method:**
- loss = L1 on **inverse** depth + gradient-matching term (the standard metric-depth fine-tuning recipe — inverse depth balances near/far pixels, the gradient term sharpens object boundaries)
- by default only the **depth head** trains (encoder frozen) — fast, adapts the metric calibration from Hypersim to NYU's Kinect geometry; set `FREEZE_ENCODER = False` to also train the encoder at a smaller LR
- a held-out slice of the streamed train samples (last 8%) is the checkpoint-selection set — the real validation split stays untouched until the final evaluation
- best checkpoint by held-out AbsRel is saved as a normal HF checkpoint dir (reusable by any `transformers` code, including the SAM2 fusion pipeline)

**Before running:** enable **GPU** (Settings → Accelerator → GPU T4) and **Internet** (Settings → Internet → On). Expect ~25–40 min: most of it is streaming the NYU train split (one-time download per session).

The pipeline scripts (`eval_nyu_depth.py`, `compare_depth_models.py`, `finetune_depth_model.py`) are embedded as `%%writefile` cells, so the notebook is self-contained. To fine-tune the **Base** or **Large** variant next, duplicate this notebook and change `MODEL_ID` / the output dirs in the Configuration cell.


## 1. Install dependencies (only what's missing)

Kaggle preinstalls numpy/scipy/torch/opencv/matplotlib/pillow as a **matched set** — blindly upgrading them (especially numpy) breaks the prebuilt scipy and torch with `ImportError`s deep inside `datasets`. The only real requirement is `transformers >= 4.48` (for Depth Pro), which recent Kaggle images already satisfy, plus `datasets`. This cell checks first and installs only if needed — on a current Kaggle image it does nothing.

In [ ]:
import subprocess
import sys
from importlib.metadata import version, PackageNotFoundError


def pkg_version(pkg):
    try:
        return tuple(int(p) for p in version(pkg).split(".")[:2])
    except PackageNotFoundError:
        return (0, 0)


# Depth Pro (apple/depth-pro-hf) needs transformers >= 4.48. DO NOT upgrade
# numpy/pillow/opencv/matplotlib here -- Kaggle's prebuilt scipy/torch are
# matched to the preinstalled versions and break if they change.
if pkg_version("transformers") < (4, 48):
    print("upgrading transformers (Depth Pro needs >= 4.48)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.48"], check=True)
    print("transformers was upgraded -- RESTART the session (Run > Restart and Clear Outputs), then re-run all cells.")

if pkg_version("datasets") == (0, 0):
    print("installing datasets...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)

import transformers, datasets
print("transformers:", transformers.__version__, "| datasets:", datasets.__version__)

## 2. Write the pipeline scripts

`eval_nyu_depth.py` (data loading + metrics) and `compare_depth_models.py` (model registry + comparison outputs) are byte-identical to the comparison notebook's, so numbers are directly comparable. `finetune_depth_model.py` is new: the training loop + an evaluation function that reuses the comparison notebook's exact protocol.

In [ ]:
%%writefile eval_nyu_depth.py
#!/usr/bin/env python3
"""
eval_nyu_depth.py

Evaluates Depth Anything V2 against NYU Depth V2 ground-truth depth maps.
For each test image: runs Depth Anything V2 (a METRIC checkpoint, so
predictions are already in meters -- no relative-to-metric scale-fitting
needed), compares against the ground-truth Kinect depth map, and reports
the standard monocular depth estimation metrics used in the literature:
AbsRel, SqRel, RMSE, RMSE(log), and the threshold accuracies delta1/2/3.

-------------------------------------------------------------------------
WHY A METRIC CHECKPOINT (not the default relative one)
-------------------------------------------------------------------------
The regular "depth-anything/Depth-Anything-V2-*-hf" checkpoints output
*relative* depth (disparity, unitless) -- comparing that directly against
NYU's metric (meters) ground truth would require a scale+shift alignment
per image, which is how relative-depth papers evaluate but adds a layer
of ambiguity. Depth Anything V2 also ships checkpoints fine-tuned for
METRIC depth on indoor scenes (Hypersim), which output real distances in
meters directly -- a much more direct, honest comparison against NYU-V2.
That's why this script defaults to:
    depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf
(swap "-Small-" for "-Base-" or "-Large-" for better accuracy/slower)

-------------------------------------------------------------------------
USAGE
-------------------------------------------------------------------------
python eval_nyu_depth.py --num-samples 200 --output-dir eval_outputs
python eval_nyu_depth.py --num-samples 50 --model depth-anything/Depth-Anything-V2-Metric-Indoor-Large-hf
"""

import argparse
import os
from dataclasses import dataclass
from typing import List, Dict, Tuple

import numpy as np
import cv2
import matplotlib.pyplot as plt


# =========================================================================
# Step 1: load NYU Depth V2 (preprocessed HF version: 'image' + 'depth_map')
# =========================================================================
def load_nyu_depth_v2(split: str, num_samples: int):
    """
    Loads the NYU Depth V2 dataset from the Hugging Face Hub.
    Each item has:
      - "image": PIL RGB image (uint8), 640x480
      - "depth_map": PIL TIFF image (float32), same size, values in meters
    """
    from datasets import load_dataset

    try:
        ds = load_dataset("sayakpaul/nyu_depth_v2", split=split, streaming=True)
    except RuntimeError:
        # datasets>=3.0 no longer runs dataset loading scripts, and this
        # dataset is script-based, so the call above raises
        # "Dataset scripts are no longer supported". Fall back to the Hub's
        # auto-converted parquet export, where the splits live under
        # default/partial-train / default/partial-validation.
        data_files = {split: f"hf://datasets/sayakpaul/nyu_depth_v2@refs/convert/parquet/default/partial-{split}/*.parquet"}
        ds = load_dataset("parquet", data_files=data_files, split=split, streaming=True)
    samples = []
    for i, item in enumerate(ds):
        if i >= num_samples:
            break
        image = np.array(item["image"].convert("RGB"))
        depth_gt = np.array(item["depth_map"]).astype(np.float32)
        samples.append((image, depth_gt))
    if not samples:
        raise RuntimeError(
            f"No samples loaded from split='{split}'. Try split='train' or split='validation'."
        )
    return samples


# =========================================================================
# Step 2: run Depth Anything V2 (metric checkpoint -> output already meters)
# =========================================================================
def load_metric_depth_model(model_id: str, device: str):
    from transformers import pipeline
    return pipeline(task="depth-estimation", model=model_id, device=0 if device == "cuda" else -1)


def predict_depth_meters(depth_pipe, image_rgb: np.ndarray) -> np.ndarray:
    """Returns predicted depth in meters, resized to match the input image."""
    from PIL import Image
    pil_image = Image.fromarray(image_rgb)
    result = depth_pipe(pil_image)

    pred = result["predicted_depth"]
    pred = pred.squeeze().detach().cpu().numpy().astype(np.float32)

    if pred.shape != image_rgb.shape[:2]:
        pred = cv2.resize(pred, (image_rgb.shape[1], image_rgb.shape[0]), interpolation=cv2.INTER_CUBIC)
    return pred


# =========================================================================
# Step 3: standard monocular depth metrics
# =========================================================================
@dataclass
class DepthMetrics:
    abs_rel: float
    sq_rel: float
    rmse: float
    rmse_log: float
    delta1: float
    delta2: float
    delta3: float


def compute_depth_metrics(pred: np.ndarray, gt: np.ndarray, min_depth: float = 1e-3, max_depth: float = 10.0) -> DepthMetrics:
    """
    Standard metrics used across the monocular depth estimation literature
    (Eigen et al. 2014 and used by every paper since, including Depth
    Anything V2's own eval protocol).

    - AbsRel: mean(|pred-gt| / gt)                  -- lower is better
    - SqRel:  mean((pred-gt)^2 / gt)                 -- lower is better
    - RMSE:   sqrt(mean((pred-gt)^2))                -- lower is better, meters
    - RMSE(log): sqrt(mean((log(pred)-log(gt))^2))   -- lower is better
    - delta_k: fraction of pixels where
               max(pred/gt, gt/pred) < 1.25^k         -- higher is better (0-1)
    """
    valid = (gt > min_depth) & (gt < max_depth) & np.isfinite(gt) & np.isfinite(pred) & (pred > min_depth)
    pred_v = pred[valid]
    gt_v = gt[valid]

    if pred_v.size == 0:
        raise ValueError("No valid pixels to evaluate -- check depth ranges / masks.")

    abs_rel = float(np.mean(np.abs(pred_v - gt_v) / gt_v))
    sq_rel = float(np.mean(((pred_v - gt_v) ** 2) / gt_v))
    rmse = float(np.sqrt(np.mean((pred_v - gt_v) ** 2)))
    rmse_log = float(np.sqrt(np.mean((np.log(pred_v) - np.log(gt_v)) ** 2)))

    ratio = np.maximum(pred_v / gt_v, gt_v / pred_v)
    delta1 = float(np.mean(ratio < 1.25))
    delta2 = float(np.mean(ratio < 1.25 ** 2))
    delta3 = float(np.mean(ratio < 1.25 ** 3))

    return DepthMetrics(abs_rel, sq_rel, rmse, rmse_log, delta1, delta2, delta3)


def average_metrics(all_metrics: List[DepthMetrics]) -> DepthMetrics:
    fields = ["abs_rel", "sq_rel", "rmse", "rmse_log", "delta1", "delta2", "delta3"]
    avg = {f: float(np.mean([getattr(m, f) for m in all_metrics])) for f in fields}
    return DepthMetrics(**avg)


def print_metrics_table(per_sample: List[DepthMetrics], overall: DepthMetrics):
    print(f"\n{'#':<5}{'AbsRel':<10}{'SqRel':<10}{'RMSE':<10}{'RMSElog':<10}{'d1':<8}{'d2':<8}{'d3':<8}")
    for i, m in enumerate(per_sample):
        print(f"{i:<5}{m.abs_rel:<10.4f}{m.sq_rel:<10.4f}{m.rmse:<10.4f}{m.rmse_log:<10.4f}{m.delta1:<8.3f}{m.delta2:<8.3f}{m.delta3:<8.3f}")
    print("-" * 70)
    print(
        f"{'AVG':<5}{overall.abs_rel:<10.4f}{overall.sq_rel:<10.4f}{overall.rmse:<10.4f}"
        f"{overall.rmse_log:<10.4f}{overall.delta1:<8.3f}{overall.delta2:<8.3f}{overall.delta3:<8.3f}"
    )
    print(
        "\nAbsRel/SqRel/RMSE/RMSElog: lower is better.  "
        "delta1/2/3: higher is better (fraction of pixels within 1.25/1.25^2/1.25^3 of ground truth)."
    )


# =========================================================================
# Step 4: visualization (rgb | ground truth | prediction | error map)
# =========================================================================
def colorize(depth: np.ndarray, vmin=None, vmax=None, cmap_name="inferno") -> np.ndarray:
    vmin = depth.min() if vmin is None else vmin
    vmax = depth.max() if vmax is None else vmax
    norm = np.clip((depth - vmin) / (vmax - vmin + 1e-8), 0, 1)
    return (plt.get_cmap(cmap_name)(norm)[:, :, :3] * 255).astype(np.uint8)


def save_comparison_figure(image_rgb, gt, pred, metrics: DepthMetrics, save_path: str):
    vmin, vmax = float(gt.min()), float(gt.max())
    error = np.abs(pred - gt)

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(image_rgb); axes[0].set_title("RGB input"); axes[0].axis("off")
    axes[1].imshow(colorize(gt, vmin, vmax)); axes[1].set_title("Ground truth (NYU-V2)"); axes[1].axis("off")
    axes[2].imshow(colorize(pred, vmin, vmax)); axes[2].set_title("Depth Anything V2 (metric)"); axes[2].axis("off")
    im = axes[3].imshow(error, cmap="viridis")
    axes[3].set_title(f"|Error|  (AbsRel={metrics.abs_rel:.3f}, RMSE={metrics.rmse:.3f}m)")
    axes[3].axis("off")
    fig.colorbar(im, ax=axes[3], fraction=0.046, pad=0.04, label="meters")

    plt.tight_layout()
    fig.savefig(save_path, dpi=130, bbox_inches="tight")
    plt.close(fig)


# =========================================================================
# Main
# =========================================================================
def main():
    parser = argparse.ArgumentParser(description="Evaluate Depth Anything V2 on NYU Depth V2")
    parser.add_argument("--split", default="validation", help="'validation' or 'train'")
    parser.add_argument("--num-samples", type=int, default=100, help="How many test images to evaluate on")
    parser.add_argument("--model", default="depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf",
                         help="HF model id -- use a Metric-Indoor checkpoint for direct meter comparison")
    parser.add_argument("--output-dir", default="eval_outputs")
    parser.add_argument("--num-visualizations", type=int, default=8, help="How many side-by-side figures to save")
    parser.add_argument("--device", default=None)
    parser.add_argument("--max-depth", type=float, default=10.0, help="NYU-V2 indoor scenes cap around here (meters)")
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)
    vis_dir = os.path.join(args.output_dir, "comparisons")
    os.makedirs(vis_dir, exist_ok=True)

    if args.device:
        device = args.device
    else:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    print(f"Loading {args.num_samples} samples from NYU Depth V2 ({args.split} split)...")
    samples = load_nyu_depth_v2(args.split, args.num_samples)
    print(f"Loaded {len(samples)} samples.")

    print(f"Loading Depth Anything V2 metric model: {args.model}")
    depth_pipe = load_metric_depth_model(args.model, device)

    per_sample_metrics = []
    for i, (image_rgb, gt_depth) in enumerate(samples):
        pred_depth = predict_depth_meters(depth_pipe, image_rgb)
        metrics = compute_depth_metrics(pred_depth, gt_depth, max_depth=args.max_depth)
        per_sample_metrics.append(metrics)

        if i < args.num_visualizations:
            save_comparison_figure(image_rgb, gt_depth, pred_depth, metrics,
                                    os.path.join(vis_dir, f"sample_{i:03d}.png"))

        if (i + 1) % 10 == 0 or (i + 1) == len(samples):
            print(f"  evaluated {i + 1}/{len(samples)}")

    overall = average_metrics(per_sample_metrics)
    print_metrics_table(per_sample_metrics, overall)

    # Save a CSV of per-sample + overall metrics for later analysis
    csv_path = os.path.join(args.output_dir, "metrics.csv")
    with open(csv_path, "w") as f:
        f.write("sample,abs_rel,sq_rel,rmse,rmse_log,delta1,delta2,delta3\n")
        for i, m in enumerate(per_sample_metrics):
            f.write(f"{i},{m.abs_rel},{m.sq_rel},{m.rmse},{m.rmse_log},{m.delta1},{m.delta2},{m.delta3}\n")
        f.write(f"AVG,{overall.abs_rel},{overall.sq_rel},{overall.rmse},{overall.rmse_log},{overall.delta1},{overall.delta2},{overall.delta3}\n")

    print(f"\nSaved per-sample metrics to: {csv_path}")
    print(f"Saved {min(args.num_visualizations, len(samples))} comparison figures to: {vis_dir}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile compare_depth_models.py
#!/usr/bin/env python3
"""
compare_depth_models.py

Runs MULTIPLE monocular depth-estimation models on the SAME NYU Depth V2
samples and produces side-by-side error metrics + comparison graphs, so
the models can be compared for the thesis.

Models compared (5 metric + 2 relative):
  - Depth Anything V2 Metric Indoor (Small / Base / Large)  -> meters directly
  - ZoeDepth (nyu-kitti)                                    -> meters directly
  - Apple Depth Pro                                          -> meters directly
  - Depth Anything V2 Small (relative)                       -> per-image aligned
  - MiDaS DPT-Large (Intel/dpt-large, relative)              -> per-image aligned

Relative-depth models output unitless disparity (higher = closer), so they
CANNOT be compared against NYU's meter-scale ground truth directly. Following
the standard MiDaS / Depth Anything eval protocol, we fit a per-image least-
squares scale + shift that maps the predicted disparity onto the ground-truth
inverse depth, convert back to meters, and only then compute the metrics.
Aligned models are marked "(aligned)" in every output -- they get per-image
information that metric models don't, which is why they're kept visually
separate in the graphs.

-------------------------------------------------------------------------
SETUP
-------------------------------------------------------------------------
pip install -r requirements.txt        # transformers>=4.48 needed for Depth Pro
pip install datasets

-------------------------------------------------------------------------
USAGE
-------------------------------------------------------------------------
# Compare all 7 models on 100 NYU validation images
python compare_depth_models.py

# Quick subset / smaller run
python compare_depth_models.py --models da2-metric-small zoedepth --num-samples 20

# List available model slugs
python compare_depth_models.py --list-models

Resumable: each model's metrics are written to <output-dir>/results/<slug>.json
as soon as that model finishes. Re-running the same command skips models whose
JSON already exists (pass --force to re-run everything). A model that fails
(e.g. unavailable checkpoint, transformers too old) is recorded as failed and
the remaining models still run.

Outputs (in --output-dir):
  results/<slug>.json            per-sample + average metrics for one model
  model_metrics.csv              one row per model, all metrics side by side
  comparison_bar_chart.png       THE comparison graph: one subplot per metric,
                                 one bar per model (aligned models hatched)
  per_sample_distribution.png    box plots of per-sample AbsRel and delta1
  error_grids/sample_XXX.png     per sample: GT + every model's depth map and
                                 |error| map on shared color scales
"""

import argparse
import json
import os
from typing import Any, Dict, List, Optional

import numpy as np
import matplotlib.pyplot as plt

# Reuse the single-model evaluation machinery -- eval_nyu_depth.py is
# import-safe (its argparse only runs under __main__).
from eval_nyu_depth import (
    DepthMetrics,
    average_metrics,
    colorize,
    compute_depth_metrics,
    load_metric_depth_model,
    load_nyu_depth_v2,
    predict_depth_meters,
)


# =========================================================================
# Model registry
# =========================================================================
# align=False -> model outputs metric depth in meters, evaluated directly.
# align=True  -> model outputs relative disparity, needs per-image
#                scale+shift alignment before metrics (MiDaS protocol).
MODEL_REGISTRY: List[Dict[str, Any]] = [
    {"slug": "da2-metric-small", "id": "depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf",
     "name": "DA V2 Metric Small", "align": False},
    {"slug": "da2-metric-base", "id": "depth-anything/Depth-Anything-V2-Metric-Indoor-Base-hf",
     "name": "DA V2 Metric Base", "align": False},
    {"slug": "da2-metric-large", "id": "depth-anything/Depth-Anything-V2-Metric-Indoor-Large-hf",
     "name": "DA V2 Metric Large", "align": False},
    {"slug": "zoedepth", "id": "Intel/zoedepth-nyu-kitti",
     "name": "ZoeDepth (nyu-kitti)", "align": False},
    {"slug": "depth-pro", "id": "apple/depth-pro-hf",
     "name": "Depth Pro", "align": False},
    {"slug": "da2-relative-small", "id": "depth-anything/Depth-Anything-V2-Small-hf",
     "name": "DA V2 Relative Small", "align": True},
    {"slug": "midas-dpt-large", "id": "Intel/dpt-large",
     "name": "MiDaS DPT-Large", "align": True},
]

METRIC_FIELDS = ["abs_rel", "sq_rel", "rmse", "rmse_log", "delta1", "delta2", "delta3"]
# (field, short label, lower-is-better?)
METRIC_SPECS = [
    ("abs_rel", "AbsRel", True),
    ("sq_rel", "SqRel", True),
    ("rmse", "RMSE (m)", True),
    ("rmse_log", "RMSE(log)", True),
    ("delta1", "delta1", False),
    ("delta2", "delta2", False),
    ("delta3", "delta3", False),
]


def registry_entry(slug: str) -> Dict[str, Any]:
    for m in MODEL_REGISTRY:
        if m["slug"] == slug:
            return m
    raise KeyError(f"Unknown model slug '{slug}'. Use --list-models to see options.")


# =========================================================================
# Scale+shift alignment for relative-depth models (MiDaS protocol)
# =========================================================================
def align_relative_depth(pred_disp: np.ndarray, gt_depth: np.ndarray,
                         min_depth: float = 1e-3, max_depth: float = 10.0):
    """
    Fits s, t so that s * pred_disp + t ~= 1 / gt_depth over valid pixels
    (least squares), then returns depth = 1 / (s*pred_disp + t) in meters.

    This is the standard evaluation protocol for relative-depth models
    (MiDaS, Depth Anything): the fit uses ground-truth information, which is
    why aligned models are marked in all outputs.

    Returns (aligned_depth, s, t).
    """
    valid = ((gt_depth > min_depth) & (gt_depth < max_depth)
             & np.isfinite(gt_depth) & np.isfinite(pred_disp))
    if valid.sum() < 2:
        raise ValueError("Not enough valid pixels to fit scale+shift alignment.")

    d_gt = 1.0 / gt_depth[valid].astype(np.float64)
    p = pred_disp[valid].astype(np.float64)
    A = np.stack([p, np.ones_like(p)], axis=1)
    (s, t), *_ = np.linalg.lstsq(A, d_gt, rcond=None)

    aligned_disp = np.clip(s * pred_disp.astype(np.float64) + t, 1e-6, None)
    return (1.0 / aligned_disp).astype(np.float32), float(s), float(t)


def clip_prediction(pred: np.ndarray, min_depth: float = 1e-3, max_depth: float = 10.0) -> np.ndarray:
    """
    Clips a predicted depth map to the valid depth band before computing
    metrics -- the standard protocol in the monocular-depth eval code
    (BTS / AdaBins / ZoeDepth all do this). Applied to EVERY model alike.

    Without this, aligned relative-depth predictions can produce absurd
    depths (a fitted disparity slightly below zero -> 1e6 m after inversion)
    that dominate RMSE/AbsRel.
    """
    return np.clip(pred, min_depth, max_depth)


# =========================================================================
# Result (de)serialization -- plain dicts in JSON so runs are resumable
# =========================================================================
def _metrics_to_dict(m: DepthMetrics) -> Dict[str, float]:
    return {f: float(getattr(m, f)) for f in METRIC_FIELDS}


def result_path(output_dir: str, slug: str) -> str:
    return os.path.join(output_dir, "results", f"{slug}.json")


def save_result(output_dir: str, entry: Dict[str, Any], result: Dict[str, Any]):
    os.makedirs(os.path.join(output_dir, "results"), exist_ok=True)
    result = dict(result, slug=entry["slug"], id=entry["id"],
                  name=entry["name"], align=entry["align"])
    with open(result_path(output_dir, entry["slug"]), "w") as f:
        json.dump(result, f, indent=2)


def load_result(output_dir: str, slug: str) -> Optional[Dict[str, Any]]:
    path = result_path(output_dir, slug)
    if not os.path.exists(path):
        return None
    with open(path) as f:
        return json.load(f)


def load_all_results(output_dir: str) -> Dict[str, Dict[str, Any]]:
    """
    Loads every results/<slug>.json in output_dir: registry models first (in
    registry order), then any EXTRA result files (e.g. the SAM2-fused model
    saved by eval_segmentation_depth.py), so the combined CSV + charts
    include them alongside the depth-only models.
    """
    results: Dict[str, Dict[str, Any]] = {}
    for e in MODEL_REGISTRY:
        r = load_result(output_dir, e["slug"])
        if r is not None:
            results[e["slug"]] = r
    results_dir = os.path.join(output_dir, "results")
    if os.path.isdir(results_dir):
        for fn in sorted(os.listdir(results_dir)):
            if not fn.endswith(".json"):
                continue
            slug = fn[: -len(".json")]
            if slug in results:
                continue
            with open(os.path.join(results_dir, fn)) as f:
                results[slug] = json.load(f)
    return results


# =========================================================================
# Core loop: one model at a time over the same sample set
# =========================================================================
def _free_gpu_memory():
    import gc
    import torch
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def run_comparison(
    models: List[Dict[str, Any]],
    num_samples: int = 100,
    split: str = "validation",
    output_dir: str = "depth_model_comparison",
    device: Optional[str] = None,
    max_depth: float = 10.0,
    num_error_grids: int = 3,
    force: bool = False,
) -> Dict[str, Dict[str, Any]]:
    """
    Evaluates each model on the SAME num_samples NYU images. Returns a dict
    slug -> result (the same dicts saved to results/<slug>.json).
    """
    os.makedirs(output_dir, exist_ok=True)

    if device is None:
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # Load the dataset ONCE so every model sees the identical sample set
    # (essential for a fair comparison).
    print(f"Loading {num_samples} NYU Depth V2 samples ({split} split) -- shared by all models...")
    samples = load_nyu_depth_v2(split, num_samples)
    num_samples = len(samples)
    print(f"Loaded {num_samples} samples.")

    results: Dict[str, Dict[str, Any]] = {}
    for entry in models:
        slug = entry["slug"]
        kind = "relative (aligned)" if entry["align"] else "metric"

        if not force:
            cached = load_result(output_dir, slug)
            if cached is not None:
                print(f"\n[{slug}] already evaluated (status: {cached.get('status')}), skipping. "
                      f"Use --force to re-run.")
                results[slug] = cached
                continue

        print(f"\n{'=' * 70}\n[{slug}] {entry['name']} ({kind})\n  model id: {entry['id']}")
        try:
            depth_pipe = load_metric_depth_model(entry["id"], device)
        except Exception as e:
            results[slug] = {"status": "failed", "error": f"failed to load model: {e}"}
            save_result(output_dir, entry, results[slug])
            print(f"  FAILED to load, continuing with other models.\n  error: {e}")
            continue

        per_sample: List[DepthMetrics] = []
        grid_preds: List[np.ndarray] = []   # predictions for the first few samples, for error grids
        try:
            for i, (image_rgb, gt_depth) in enumerate(samples):
                pred = predict_depth_meters(depth_pipe, image_rgb)
                if entry["align"]:
                    pred, s, t = align_relative_depth(pred, gt_depth, max_depth=max_depth)
                pred = clip_prediction(pred, max_depth=max_depth)
                metrics = compute_depth_metrics(pred, gt_depth, max_depth=max_depth)
                per_sample.append(metrics)

                if i < num_error_grids:
                    grid_preds.append(pred)

                if (i + 1) % 10 == 0 or (i + 1) == num_samples:
                    print(f"  evaluated {i + 1}/{num_samples}")
        except Exception as e:
            results[slug] = {"status": "failed", "error": str(e), "num_samples_evaluated": len(per_sample)}
            save_result(output_dir, entry, results[slug])
            print(f"  FAILED mid-run, continuing with other models.\n  error: {e}")
            _free_gpu_memory()
            continue

        avg = average_metrics(per_sample)
        results[slug] = {
            "status": "ok",
            "num_samples": num_samples,
            "max_depth": max_depth,
            "per_sample": [_metrics_to_dict(m) for m in per_sample],
            "average": _metrics_to_dict(avg),
        }
        save_result(output_dir, entry, results[slug])
        print(f"  done. Avg AbsRel={avg.abs_rel:.4f}  RMSE={avg.rmse:.3f}m  delta1={avg.delta1:.3f}")

        if grid_preds:
            np.save(os.path.join(output_dir, "results", f"{slug}_grid_preds.npy"),
                    np.stack(grid_preds).astype(np.float32))

        del depth_pipe
        _free_gpu_memory()

    return results


# =========================================================================
# Combined outputs: CSV + comparison graphs
# =========================================================================
def write_metrics_csv(results: Dict[str, Dict[str, Any]], output_dir: str):
    path = os.path.join(output_dir, "model_metrics.csv")
    with open(path, "w", newline="") as f:
        f.write("slug,name,kind,status," + ",".join(METRIC_FIELDS) + "\n")
        for entry in _entries_with_extras(results):
            r = results[entry["slug"]]
            kind = "relative+aligned" if entry["align"] else "metric"
            if r.get("status") != "ok":
                f.write(f"{entry['slug']},{entry['name']},{kind},failed,\n")
                continue
            avg = r["average"]
            vals = ",".join(f"{avg[k]:.6f}" for k in METRIC_FIELDS)
            f.write(f"{entry['slug']},{entry['name']},{kind},ok,{vals}\n")
    print(f"Saved combined metrics CSV to: {path}")


def _entries_with_extras(results: Dict[str, Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Registry entries present in results (registry order first), followed by
    entries for extra result files not in the registry (e.g. the SAM2-fused
    model), constructed from the metadata each result saved about itself.
    """
    entries = [e for e in MODEL_REGISTRY if e["slug"] in results]
    known = {e["slug"] for e in entries}
    for slug, r in results.items():
        if slug in known:
            continue
        entries.append({"slug": slug, "id": r.get("id", ""),
                        "name": r.get("name", slug), "align": bool(r.get("align", False))})
    return entries


def _ok_entries(results: Dict[str, Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Entries (registry + extras) that have a successful result."""
    return [e for e in _entries_with_extras(results)
            if results[e["slug"]].get("status") == "ok"]


def _bar_colors(entries: List[Dict[str, Any]]):
    cmap = plt.get_cmap("tab10")
    return [cmap(i % 10) for i in range(len(entries))]


def plot_comparison_bar_chart(results: Dict[str, Dict[str, Any]], output_dir: str):
    """
    The main comparison graph: one subplot per metric, one bar per model.
    Aligned (relative) models get a hatch so the protocol difference is visible.
    """
    entries = _ok_entries(results)
    if not entries:
        print("No successful model results -- skipping bar chart.")
        return
    colors = _bar_colors(entries)
    names = [f"{e['name']}\n(aligned)" if e["align"] else e["name"] for e in entries]

    fig, axes = plt.subplots(2, 4, figsize=(22, 10))
    axes = axes.ravel()

    for ax, (field, label, lower_better) in zip(axes, METRIC_SPECS):
        vals = [results[e["slug"]]["average"][field] for e in entries]
        bars = ax.bar(range(len(entries)), vals, color=colors)
        for bar, entry, v in zip(bars, entries, vals):
            if entry["align"]:
                bar.set_hatch("//")
                bar.set_edgecolor("white")
        ax.set_title(f"{label}  ({'lower is better' if lower_better else 'higher is better'})",
                     fontsize=12)
        ax.set_xticks(range(len(entries)))
        ax.set_xticklabels(names, fontsize=8, rotation=45, ha="right")
        # value labels on top of each bar
        vmax = max(vals) if vals else 1.0
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    f"{v:.3f}", ha="center", va="bottom", fontsize=8)
        ax.set_ylim(0, vmax * 1.15 if vmax > 0 else 1.0)
        ax.spines[["top", "right"]].set_visible(False)

    axes[-1].axis("off")
    axes[-1].text(0.02, 0.95,
                  "Hatched = relative-depth model,\nevaluated with per-image\nscale+shift alignment\n(MiDaS protocol: uses GT info,\nnot directly comparable to\nmetric models)",
                  transform=axes[-1].transAxes, fontsize=10, va="top")

    fig.suptitle("Depth model comparison on NYU Depth V2 "
                 f"({len(entries)} models, {results[entries[0]['slug']]['num_samples']} images)", fontsize=15)
    plt.tight_layout()
    path = os.path.join(output_dir, "comparison_bar_chart.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved comparison bar chart to: {path}")


def plot_per_sample_distribution(results: Dict[str, Dict[str, Any]], output_dir: str):
    """Box plots of per-sample AbsRel and delta1 -- shows variance, not just means."""
    entries = _ok_entries(results)
    if not entries:
        print("No successful model results -- skipping distribution plot.")
        return
    colors = _bar_colors(entries)
    names = [f"{e['name']}\n(aligned)" if e["align"] else e["name"] for e in entries]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    for ax, (field, label, lower_better) in zip(axes, [("abs_rel", "AbsRel", True), ("delta1", "delta1", False)]):
        data = [results[e["slug"]]["per_sample"] for e in entries]
        data = [[s[field] for s in per_model] for per_model in data]
        boxes = ax.boxplot(data, patch_artist=True, showfliers=True,
                           flierprops=dict(markersize=3, alpha=0.5))
        ax.set_xticks(range(1, len(entries) + 1))
        ax.set_xticklabels(names)
        for box, entry, color in zip(boxes["boxes"], entries, colors):
            box.set_facecolor(color)
            if entry["align"]:
                box.set_hatch("//")
                box.set_edgecolor("white")
        ax.set_title(f"Per-sample {label} distribution ({'lower is better' if lower_better else 'higher is better'})",
                     fontsize=12)
        ax.tick_params(axis="x", labelsize=8, rotation=45)
        ax.spines[["top", "right"]].set_visible(False)

    fig.suptitle("Per-sample metric distributions on NYU Depth V2", fontsize=14)
    plt.tight_layout()
    path = os.path.join(output_dir, "per_sample_distribution.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved per-sample distribution plot to: {path}")


def plot_error_grids(results: Dict[str, Dict[str, Any]], samples, output_dir: str, num_grids: int):
    """
    For the first num_grids samples: one row per model, showing its depth map
    and its |error| map, both on color scales shared across all models, with
    the ground truth on top. Uses the <slug>_grid_preds.npy files saved during
    each model's run (so this works across resumed runs too).
    """
    if num_grids <= 0 or not samples:
        return
    entries = _ok_entries(results)
    if not entries:
        print("No successful model results -- skipping error grids.")
        return

    preds = {}
    for e in entries:
        path = os.path.join(output_dir, "results", f"{e['slug']}_grid_preds.npy")
        if os.path.exists(path):
            preds[e["slug"]] = np.load(path)

    have = [e for e in entries if e["slug"] in preds and len(preds[e["slug"]]) >= min(num_grids, len(samples))]
    if not have:
        print("No cached grid predictions found (they're only saved during a fresh model run) -- skipping error grids.")
        return

    grids_dir = os.path.join(output_dir, "error_grids")
    os.makedirs(grids_dir, exist_ok=True)

    for i in range(min(num_grids, len(samples))):
        image_rgb, gt = samples[i]
        vmin, vmax = float(gt.min()), float(gt.max())

        # shared error scale: max |error| over all models for this sample
        err_maps = {e["slug"]: np.abs(preds[e["slug"]][i] - gt) for e in have}
        err_vmax = max(float(m.max()) for m in err_maps.values())

        n_rows = len(have) + 1  # + GT row
        fig, axes = plt.subplots(n_rows, 2, figsize=(10, 2.1 * n_rows),
                                 gridspec_kw={"width_ratios": [1, 1]})
        if n_rows == 2:
            axes = axes.reshape(2, 2)

        # Row 0: ground truth
        axes[0, 0].imshow(colorize(gt, vmin, vmax))
        axes[0, 0].set_ylabel("Ground truth", fontsize=10)
        axes[0, 1].axis("off")
        axes[0, 1].text(0.5, 0.5, "reference\n(see colorbar)", ha="center", va="center", fontsize=10)

        # One row per model: depth prediction | |error| map
        for r, e in enumerate(have, start=1):
            pred = preds[e["slug"]][i]
            m = results[e["slug"]]["per_sample"][i]
            label = f"{e['name']}" + (" (aligned)" if e["align"] else "")
            axes[r, 0].imshow(colorize(pred, vmin, vmax))
            axes[r, 0].set_ylabel(f"{label}\nAbsRel={m['abs_rel']:.3f}", fontsize=9)
            im = axes[r, 1].imshow(err_maps[e["slug"]], cmap="viridis", vmin=0, vmax=err_vmax)
            if r == len(have):
                fig.colorbar(im, ax=axes[:, 1], fraction=0.03, pad=0.02, label="|error| (m)")

        for ax in axes.ravel():
            ax.set_xticks([]); ax.set_yticks([])
        for c, title in enumerate(["depth (m)", "|error| (m)"]):
            axes[0, c].set_title(title, fontsize=11)

        fig.suptitle(f"NYU sample {i}: model predictions vs ground truth", fontsize=13)
        path = os.path.join(grids_dir, f"sample_{i:03d}.png")
        fig.savefig(path, dpi=130, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved error grid to: {path}")


# =========================================================================
# Main
# =========================================================================
def main():
    parser = argparse.ArgumentParser(description="Compare multiple depth-estimation models on NYU Depth V2")
    parser.add_argument("--models", nargs="*", default=None,
                        help="Model slugs to evaluate (default: all). See --list-models.")
    parser.add_argument("--split", default="validation", help="'validation' or 'train'")
    parser.add_argument("--num-samples", type=int, default=100, help="How many NYU images per model")
    parser.add_argument("--output-dir", default="depth_model_comparison")
    parser.add_argument("--device", default=None, help="cuda | cpu (auto-detected if omitted)")
    parser.add_argument("--max-depth", type=float, default=10.0, help="Valid GT depth range cap (meters)")
    parser.add_argument("--num-error-grids", type=int, default=3,
                        help="How many per-sample error-grid figures to save")
    parser.add_argument("--force", action="store_true", help="Re-run models even if a saved result exists")
    parser.add_argument("--list-models", action="store_true", help="Print the model registry and exit")
    args = parser.parse_args()

    if args.list_models:
        print(f"{'slug':<22}{'name':<28}{'kind':<20}id")
        for e in MODEL_REGISTRY:
            kind = "relative (aligned)" if e["align"] else "metric"
            print(f"{e['slug']:<22}{e['name']:<28}{kind:<20}{e['id']}")
        return

    if args.models:
        models = [registry_entry(s) for s in args.models]
    else:
        models = MODEL_REGISTRY

    results = run_comparison(
        models=models,
        num_samples=args.num_samples,
        split=args.split,
        output_dir=args.output_dir,
        device=args.device,
        max_depth=args.max_depth,
        num_error_grids=args.num_error_grids,
        force=args.force,
    )

    # Combined outputs. NOTE: bar chart / CSV include ALL saved results in the
    # output dir (including extra models like the SAM2-fused one), not just
    # the ones from this invocation, so a resumed run still produces a
    # complete comparison.
    all_results = load_all_results(args.output_dir)

    write_metrics_csv(all_results, args.output_dir)
    plot_comparison_bar_chart(all_results, args.output_dir)
    plot_per_sample_distribution(all_results, args.output_dir)

    # reload samples for the error grids (cheap; keeps run_comparison's memory flat)
    grid_samples = load_nyu_depth_v2(args.split, args.num_error_grids) if args.num_error_grids > 0 else []
    plot_error_grids(all_results, grid_samples, args.output_dir, args.num_error_grids)

    ok = sum(1 for r in all_results.values() if r.get("status") == "ok")
    failed = [s for s, r in all_results.items() if r.get("status") != "ok"]
    print(f"\nComparison complete: {ok} models ok, {len(failed)} failed"
          + (f" ({', '.join(failed)})" if failed else ""))
    print(f"All outputs in: {os.path.abspath(args.output_dir)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile finetune_depth_model.py
#!/usr/bin/env python3
"""
finetune_depth_model.py

Fine-tunes a Depth Anything V2 METRIC checkpoint (the transformers-native
depth-anything/Depth-Anything-V2-Metric-Indoor-*-hf models) on the NYU
Depth V2 train split, and evaluates the result with the EXACT same protocol
as compare_depth_models.py -- same validation samples, same 7 metrics,
same valid-pixel mask, same depth clipping, same results/<slug>.json format
-- so the fine-tuned model can be compared head-to-head against the
pretrained baseline (and dropped into the comparison notebook's charts).

WHAT IS TRAINED
---------------
The model is a DINOv2 encoder + metric depth head. By default only the
depth head is trained (freeze_encoder=True): that adapts the model's metric
calibration from Hypersim (its original indoor training set) to NYU's
Kinect geometry, is fast, and cannot catastrophically forget the encoder's
features. Set freeze_encoder=False to also fine-tune the encoder at a much
smaller learning rate (the classic "head-only vs full fine-tuning" trade).

LOSS
----
L1 on INVERSE depth over valid pixels (0 < gt < max_depth) plus a
gradient-matching term on inverse depth:
    loss = |1/pred - 1/gt|_1 + lambda_grad * |grad(1/pred) - grad(1/gt)|_1
Inverse depth balances near/far pixels (plain L1 on meters is dominated by
the far side of the room) and the gradient term sharpens object boundaries.
This is the standard recipe for metric-depth fine-tuning, used in varying
forms by MiDaS / Depth Anything / ZoeDepth.

CHECKPOINT SELECTION
--------------------
A held-out slice of the streamed train samples (val_fraction) is evaluated
every eval_every optimizer steps; the best checkpoint by val AbsRel is
written to output_ckpt_dir (a normal HF checkpoint dir: config.json +
preprocessor_config.json), and the training history to history.json next
to it. The held-out slice is deliberately carved from the TRAIN stream --
the actual validation split stays untouched until the final evaluation.

USAGE
-----
# Library use (from the notebook):
    from finetune_depth_model import finetune, evaluate_finetuned
    info = finetune("depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf",
                    "/kaggle/working/finetuned_da2_metric_small")
    evaluate_finetuned(info["ckpt_dir"], slug="da2-metric-small-ft", ...)

# Standalone:
    python finetune_depth_model.py --num-train-samples 600 --epochs 4

Requires eval_nyu_depth.py (data loading + metrics) and, for
evaluate_finetuned(), compare_depth_models.py in the same directory -- both
are written by the notebooks' %%writefile cells.
"""

import argparse
import json
import math
import os
import random
from typing import Any, Dict, List, Optional

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

from eval_nyu_depth import (
    average_metrics,
    compute_depth_metrics,
    load_metric_depth_model,
    load_nyu_depth_v2,
    predict_depth_meters,
)


# =========================================================================
# Reproducibility
# =========================================================================
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# =========================================================================
# Model
# =========================================================================
def build_model(model_id: str, freeze_encoder: bool = True):
    """
    Loads a Depth Anything V2 (metric) checkpoint as
    AutoModelForDepthEstimation + its image processor. With freeze_encoder,
    the DINOv2 backbone's parameters are frozen so only the depth head
    trains.
    """
    from transformers import AutoImageProcessor, AutoModelForDepthEstimation

    processor = AutoImageProcessor.from_pretrained(model_id)
    model = AutoModelForDepthEstimation.from_pretrained(model_id)

    backbone = _backbone_of(model)
    if backbone is None:
        print("WARNING: could not locate the encoder -- freeze_encoder has no effect.")
    elif freeze_encoder:
        for p in backbone.parameters():
            p.requires_grad_(False)
    return model, processor


def _backbone_of(model):
    """The DINOv2 encoder inside a DepthAnythingForDepthEstimation."""
    core = getattr(model, "model", model)
    return getattr(core, "backbone", None) or getattr(core, "encoder", None)


# =========================================================================
# Batching + augmentation (horizontal flip, applied to image AND depth)
# =========================================================================
def _flip(sample, rng):
    image, depth = sample
    if rng.random() < 0.5:
        image = np.ascontiguousarray(image[:, ::-1])
        depth = np.ascontiguousarray(depth[:, ::-1])
    return image, depth


def _minibatches(samples, batch_size, shuffle, rng):
    order = np.arange(len(samples))
    if shuffle:
        rng.shuffle(order)
    for s in range(0, len(order), batch_size):
        yield [samples[i] for i in order[s:s + batch_size]]


def _collate(batch, processor, augment, rng, device):
    if augment:
        batch = [_flip(s, rng) for s in batch]
    images = [Image.fromarray(im) for im, _ in batch]
    pixel_values = processor(images=images, return_tensors="pt")["pixel_values"].to(device)
    gt_depth = torch.from_numpy(np.stack([d.astype(np.float32) for _, d in batch])).to(device)
    return pixel_values, gt_depth


# =========================================================================
# Loss: L1 on inverse depth + gradient matching (see module docstring)
# =========================================================================
def _masked_l1(x, m):
    if bool(m.any()):
        return x[m].abs().mean()
    return x.sum() * 0.0


def _gradient_matching_loss(pred_inv, gt_inv, mask):
    # First-order image gradients, counted only where BOTH neighboring
    # pixels are valid (so mask boundaries don't leak fake gradients).
    def grads(x, m):
        gx = x[..., :, 1:] - x[..., :, :-1]
        gy = x[..., 1:, :] - x[..., :-1, :]
        mx = m[..., :, 1:] & m[..., :, :-1]
        my = m[..., 1:, :] & m[..., :-1, :]
        return gx, gy, mx, my

    pgx, pgy, pmx, pmy = grads(pred_inv, mask)
    ggx, ggy, gmx, gmy = grads(gt_inv, mask)
    return _masked_l1(pgx - ggx, pmx) + _masked_l1(pgy - ggy, pmy)


def depth_loss(pred, gt_depth, max_depth=10.0, lambda_grad=0.5, min_depth=1e-3):
    """
    pred, gt_depth: (B, H, W) tensors in meters, already on the same grid
    (the caller resizes gt onto pred's grid). Returns (total, l1, grad).
    """
    mask = (gt_depth > min_depth) & (gt_depth < max_depth) & torch.isfinite(gt_depth)
    zero = pred.sum() * 0.0
    if not bool(mask.any()):
        return zero, zero, zero

    pred_inv = 1.0 / pred.clamp(min=min_depth, max=max_depth)
    gt_inv = 1.0 / gt_depth.clamp(min=min_depth, max=max_depth)

    l1 = (pred_inv[mask] - gt_inv[mask]).abs().mean()
    grad = _gradient_matching_loss(pred_inv, gt_inv, mask) if lambda_grad > 0 else zero
    return l1 + lambda_grad * grad, l1.detach(), grad.detach()


# =========================================================================
# Validation on a held-out slice of the train samples (checkpoint selection)
# =========================================================================
@torch.no_grad()
def _evaluate_val(model, processor, samples, max_depth, device, batch_size=8):
    """
    Mean metrics with the GT resized onto the model's output grid -- used
    only for picking the best checkpoint. The final head-to-head evaluation
    goes through evaluate_finetuned() and the comparison-notebook protocol
    (full-resolution pipeline output) instead.
    """
    was_training = model.training
    model.eval()
    rng = np.random.default_rng(0)
    per_sample = []
    for batch in _minibatches(samples, batch_size, shuffle=False, rng=rng):
        pixel_values, gt = _collate(batch, processor, augment=False, rng=rng, device=device)
        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            pred = model(pixel_values=pixel_values).predicted_depth
        if pred.dim() == 4:
            pred = pred.squeeze(1)
        pred = pred.float().clamp(1e-3, max_depth)
        gt_r = F.interpolate(gt[:, None], size=pred.shape[-2:], mode="nearest")[:, 0]
        for b in range(pred.shape[0]):
            per_sample.append(compute_depth_metrics(
                pred[b].cpu().numpy(), gt_r[b].cpu().numpy(), max_depth=max_depth))
    if was_training:
        model.train()
    return average_metrics(per_sample)


# =========================================================================
# The training loop
# =========================================================================
def finetune(
    model_id: str,
    output_ckpt_dir: str,
    num_train_samples: int = 600,
    val_fraction: float = 0.08,
    epochs: int = 4,
    batch_size: int = 8,
    lr_head: float = 5e-5,
    lr_encoder: float = 1e-6,
    freeze_encoder: bool = True,
    lambda_grad: float = 0.5,
    max_depth: float = 10.0,
    device: Optional[str] = None,
    seed: int = 42,
    eval_every: int = 40,
    log_every: int = 10,
    split: str = "train",
    weight_decay: float = 0.01,
    warmup_fraction: float = 0.05,
    grad_clip: float = 1.0,
) -> Dict[str, Any]:
    """
    Fine-tunes model_id on NYU Depth V2 (train split) and returns a dict
    with the best checkpoint dir, the training history, and the val AbsRel
    of the pretrained vs. best checkpoint.
    """
    set_seed(seed)
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"device: {device}")
    rng = np.random.default_rng(seed)

    # ---------------- data ----------------
    print(f"streaming {num_train_samples} NYU Depth V2 ({split}) samples "
          f"(one-time download, takes a while)...")
    samples = load_nyu_depth_v2(split, num_train_samples)
    num_val = max(1, int(round(len(samples) * val_fraction)))
    val_samples, train_samples = samples[-num_val:], samples[:-num_val]
    print(f"split: {len(train_samples)} train / {num_val} held-out val "
          f"(last {val_fraction:.0%} of the stream -- the validation split "
          f"stays untouched until the final evaluation)")

    # ---------------- model ----------------
    model, processor = build_model(model_id, freeze_encoder=freeze_encoder)
    model.to(device)
    backbone = _backbone_of(model)
    backbone_ids = set() if backbone is None else {id(p) for p in backbone.parameters()}
    head_params = [p for p in model.parameters() if p.requires_grad and id(p) not in backbone_ids]
    enc_params = [p for p in model.parameters() if p.requires_grad and id(p) in backbone_ids]
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"trainable parameters: {n_trainable / 1e6:.2f}M "
          f"(encoder {'frozen' if freeze_encoder else 'UNFROZEN'})")

    groups = [{"params": head_params, "lr": lr_head}]
    if enc_params:
        groups.append({"params": enc_params, "lr": lr_encoder})
    optimizer = torch.optim.AdamW(groups, weight_decay=weight_decay)

    steps_per_epoch = math.ceil(len(train_samples) / batch_size)
    total_steps = epochs * steps_per_epoch
    warmup_steps = max(1, int(warmup_fraction * total_steps))

    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * min(1.0, p)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    use_amp = device == "cuda"
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except (TypeError, AttributeError):  # older torch
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    os.makedirs(output_ckpt_dir, exist_ok=True)

    def save_checkpoint():
        model.save_pretrained(output_ckpt_dir)
        processor.save_pretrained(output_ckpt_dir)

    # ------------- initial val: the pretrained model on the held-out slice
    init = _evaluate_val(model, processor, val_samples, max_depth, device, batch_size)
    best_val = init.abs_rel
    print(f"[step 0 / pretrained] val AbsRel={init.abs_rel:.4f}  RMSE={init.rmse:.3f}m")

    history: List[Dict[str, Any]] = []
    val_history: List[Dict[str, Any]] = [
        {"step": 0, "val_abs_rel": init.abs_rel, "val_rmse": init.rmse}]
    saved_any = False

    # ---------------- training ----------------
    gstep = 0
    for epoch in range(1, epochs + 1):
        model.train()
        if freeze_encoder and backbone is not None:
            backbone.eval()  # keep the frozen encoder in eval mode
        for batch in _minibatches(train_samples, batch_size, shuffle=True, rng=rng):
            pixel_values, gt = _collate(batch, processor, augment=True, rng=rng, device=device)
            with torch.amp.autocast("cuda", enabled=use_amp):
                pred = model(pixel_values=pixel_values).predicted_depth
                if pred.dim() == 4:
                    pred = pred.squeeze(1)
                # compare on the model's output grid: resize GT to the prediction
                gt_r = F.interpolate(gt[:, None], size=pred.shape[-2:], mode="nearest")[:, 0]
                loss, l1, grad = depth_loss(pred.float(), gt_r,
                                            max_depth=max_depth, lambda_grad=lambda_grad)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            gstep += 1

            if gstep % log_every == 0 or gstep == total_steps:
                history.append({"epoch": epoch, "step": gstep,
                                "loss": float(loss.detach()), "l1": float(l1),
                                "grad": float(grad), "lr": optimizer.param_groups[0]["lr"]})
                print(f"[epoch {epoch} step {gstep}/{total_steps}] "
                      f"loss={float(loss.detach()):.5f} (l1={float(l1):.5f} grad={float(grad):.5f}) "
                      f"lr={optimizer.param_groups[0]['lr']:.2e}")

            if gstep % eval_every == 0 or gstep == total_steps:
                vm = _evaluate_val(model, processor, val_samples, max_depth, device, batch_size)
                val_history.append({"step": gstep, "val_abs_rel": vm.abs_rel,
                                    "val_rmse": vm.rmse})
                marker = ""
                if vm.abs_rel < best_val:
                    best_val = vm.abs_rel
                    save_checkpoint()
                    saved_any = True
                    marker = "  ** new best, checkpoint saved"
                print(f"[epoch {epoch} step {gstep}] val AbsRel={vm.abs_rel:.4f}  "
                      f"RMSE={vm.rmse:.3f}m{marker}")

    # Always leave a usable checkpoint behind, even if training never beat
    # the pretrained initialization on the held-out slice.
    if not saved_any:
        save_checkpoint()
        print("WARNING: fine-tuning never improved on the pretrained model on the "
              "held-out slice -- the FINAL weights were saved, but by val AbsRel "
              "they are no better than the pretrained checkpoint.")

    meta = {
        "model_id": model_id,
        "freeze_encoder": freeze_encoder,
        "num_train_samples": len(train_samples),
        "num_val_samples": len(val_samples),
        "epochs": epochs,
        "batch_size": batch_size,
        "lr_head": lr_head,
        "lr_encoder": lr_encoder,
        "lambda_grad": lambda_grad,
        "max_depth": max_depth,
        "seed": seed,
        "total_steps": total_steps,
        "pretrained_val_abs_rel": init.abs_rel,
        "best_val_abs_rel": best_val,
    }
    with open(os.path.join(output_ckpt_dir, "history.json"), "w") as f:
        json.dump({"meta": meta, "history": history, "val_history": val_history}, f, indent=2)

    print(f"\nfine-tuning done: held-out val AbsRel {init.abs_rel:.4f} (pretrained) "
          f"-> {best_val:.4f} (best)")
    return {"ckpt_dir": output_ckpt_dir,
            "best_val_abs_rel": best_val,
            "pretrained_val_abs_rel": init.abs_rel,
            "history": history, "val_history": val_history, "meta": meta}


# =========================================================================
# Final evaluation -- the exact comparison-notebook protocol
# =========================================================================
def evaluate_finetuned(
    model_id_or_path: str,
    slug: str,
    name: Optional[str] = None,
    num_samples: int = 100,
    split: str = "validation",
    output_dir: str = "depth_model_comparison",
    device: Optional[str] = None,
    max_depth: float = 10.0,
    samples: Optional[list] = None,
    num_error_grids: int = 0,
    force: bool = False,
) -> Dict[str, Any]:
    """
    Evaluates a model (HF hub id OR a local fine-tuned checkpoint dir)
    through the SAME path as compare_depth_models.run_comparison: the HF
    pipeline, predict_depth_meters, clip_prediction, the same 7 metrics --
    and saves the result as results/<slug>.json in the same format, so it
    joins model_metrics.csv and all comparison charts automatically.

    Skips (and returns) the cached result if one exists, unless force=True.
    """
    import compare_depth_models as cdm

    cached = cdm.load_result(output_dir, slug)
    if cached is not None and not force:
        print(f"[{slug}] result already saved in {output_dir} (status: {cached.get('status')}), "
              f"skipping. Pass force=True to re-run.")
        return cached

    if samples is None:
        print(f"loading {num_samples} NYU Depth V2 samples ({split} split)...")
        samples = load_nyu_depth_v2(split, num_samples)
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    entry = {"slug": slug, "id": model_id_or_path,
             "name": name or slug, "align": False}
    print(f"\n[{slug}] evaluating {model_id_or_path} on {len(samples)} samples...")
    depth_pipe = load_metric_depth_model(model_id_or_path, device)

    per_sample = []
    grid_preds: List[np.ndarray] = []
    try:
        for i, (image_rgb, gt_depth) in enumerate(samples):
            pred = predict_depth_meters(depth_pipe, image_rgb)
            pred = cdm.clip_prediction(pred, max_depth=max_depth)
            per_sample.append(compute_depth_metrics(pred, gt_depth, max_depth=max_depth))
            if i < num_error_grids:
                grid_preds.append(pred)
            if (i + 1) % 10 == 0 or (i + 1) == len(samples):
                print(f"  evaluated {i + 1}/{len(samples)}")
    finally:
        del depth_pipe
        cdm._free_gpu_memory()

    avg = average_metrics(per_sample)
    result = {
        "status": "ok",
        "num_samples": len(samples),
        "max_depth": max_depth,
        "per_sample": [cdm._metrics_to_dict(m) for m in per_sample],
        "average": cdm._metrics_to_dict(avg),
    }
    cdm.save_result(output_dir, entry, result)
    if grid_preds:
        np.save(os.path.join(output_dir, "results", f"{slug}_grid_preds.npy"),
                np.stack(grid_preds).astype(np.float32))
    print(f"  done. Avg AbsRel={avg.abs_rel:.4f}  RMSE={avg.rmse:.3f}m  delta1={avg.delta1:.3f}")
    return result


# =========================================================================
# Main (standalone use)
# =========================================================================
def main():
    ap = argparse.ArgumentParser(
        description="Fine-tune a Depth Anything V2 metric checkpoint on NYU Depth V2.")
    ap.add_argument("--model", default="depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf")
    ap.add_argument("--ckpt-dir", default="finetuned_da2_metric_small")
    ap.add_argument("--num-train-samples", type=int, default=600)
    ap.add_argument("--epochs", type=int, default=4)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--lr-head", type=float, default=5e-5)
    ap.add_argument("--lr-encoder", type=float, default=1e-6)
    ap.add_argument("--unfreeze-encoder", action="store_true")
    ap.add_argument("--lambda-grad", type=float, default=0.5)
    ap.add_argument("--eval-samples", type=int, default=100)
    ap.add_argument("--slug", default="da2-metric-small-ft")
    ap.add_argument("--output-dir", default="depth_model_comparison")
    args = ap.parse_args()

    info = finetune(
        args.model, args.ckpt_dir,
        num_train_samples=args.num_train_samples, epochs=args.epochs,
        batch_size=args.batch_size, lr_head=args.lr_head, lr_encoder=args.lr_encoder,
        freeze_encoder=not args.unfreeze_encoder, lambda_grad=args.lambda_grad,
    )
    evaluate_finetuned(info["ckpt_dir"], slug=args.slug,
                       num_samples=args.eval_samples, output_dir=args.output_dir)


if __name__ == "__main__":
    main()


In [ ]:
import importlib
import inspect

import eval_nyu_depth, compare_depth_models, finetune_depth_model

# Pick up whatever is on disk right now if the modules were imported earlier.
for m in (eval_nyu_depth, compare_depth_models, finetune_depth_model):
    importlib.reload(m)

# Fail loudly on stale copies instead of dying later mid-training.
assert "partial-" in inspect.getsource(eval_nyu_depth.load_nyu_depth_v2), (
    "eval_nyu_depth.py in /kaggle/working is STALE (no parquet fallback). "
    "Re-run the %%writefile cells above."
)

print("imported:", len(compare_depth_models.MODEL_REGISTRY), "models in registry")
print("finetune_depth_model functions:",
      [f for f in ("finetune", "evaluate_finetuned", "build_model", "depth_loss")
       if callable(getattr(finetune_depth_model, f, None))])


## 3. Configuration

In [ ]:
# -----------------------------------------------------------------------
# CONFIGURATION — edit these
# -----------------------------------------------------------------------

# The model being fine-tuned (this notebook: the Small metric checkpoint)
MODEL_ID = "depth-anything/Depth-Anything-V2-Metric-Indoor-Small-hf"

# Training
NUM_TRAIN_SAMPLES = 3000   # streamed from the NYU TRAIN split (one-time download)
EPOCHS = 3
BATCH_SIZE = 8
LR_HEAD = 5e-5             # depth-head learning rate
LR_ENCODER = 1e-6          # only used when FREEZE_ENCODER = False
FREEZE_ENCODER = True      # head-only fine-tuning (see title cell)
LAMBDA_GRAD = 0.5          # weight of the gradient-matching loss term

# Evaluation (must match the comparison notebook for comparability)
NUM_EVAL_SAMPLES = 100     # same first-100 validation images as tsk3succeed
BASELINE_SLUG = "da2-metric-small"          # slug in MODEL_REGISTRY
FT_SLUG = "da2-metric-small-ft"             # result file for the fine-tuned model
FT_NAME = "DA V2 Metric Small (fine-tuned)"

# Outputs
FT_CKPT_DIR = "/kaggle/working/finetuned_da2_metric_small"
OUTPUT_DIR = "/kaggle/working/depth_model_comparison"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")


## 4. Fine-tune

Streams the NYU **train** split once (the slow part — it's a one-time download per session), carves off the last 8% as a held-out checkpoint-selection set, and trains. Every `eval_every` optimizer steps the held-out slice is evaluated; the best checkpoint by held-out AbsRel is saved to `FT_CKPT_DIR` (plus `history.json` with the full training/val curves).

In [ ]:
import finetune_depth_model as fdm

info = fdm.finetune(
    model_id=MODEL_ID,
    output_ckpt_dir=FT_CKPT_DIR,
    num_train_samples=NUM_TRAIN_SAMPLES,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr_head=LR_HEAD,
    lr_encoder=LR_ENCODER,
    freeze_encoder=FREEZE_ENCODER,
    lambda_grad=LAMBDA_GRAD,
    device=DEVICE,
)

print("\nbest checkpoint:", info["ckpt_dir"])
print(f"held-out val AbsRel: {info['pretrained_val_abs_rel']:.4f} (pretrained)"
      f" -> {info['best_val_abs_rel']:.4f} (fine-tuned)")


## 5. Training curves

In [ ]:
import json
import os

import matplotlib.pyplot as plt

hist = json.load(open(os.path.join(FT_CKPT_DIR, "history.json")))
h, v = hist["history"], hist["val_history"]
pre = hist["meta"]["pretrained_val_abs_rel"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot([x["step"] for x in h], [x["loss"] for x in h], label="total")
axes[0].plot([x["step"] for x in h], [x["l1"] for x in h], label="L1 (inverse depth)")
axes[0].plot([x["step"] for x in h], [x["grad"] for x in h], label="gradient term")
axes[0].set_xlabel("optimizer step")
axes[0].set_ylabel("train loss")
axes[0].set_title("Training loss")
axes[0].legend()

axes[1].plot([x["step"] for x in v], [x["val_abs_rel"] for x in v], marker="o", label="fine-tuned")
axes[1].axhline(pre, color="tab:red", ls="--", label=f"pretrained ({pre:.4f})")
axes[1].set_xlabel("optimizer step")
axes[1].set_ylabel("held-out AbsRel")
axes[1].set_title("Checkpoint-selection metric (lower is better)")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(FT_CKPT_DIR, "training_curves.png"), dpi=150, bbox_inches="tight")
plt.show()


## 6. Evaluate with the comparison protocol

First makes sure the **pretrained baseline** has a result in `OUTPUT_DIR` (skipped instantly if it's already there — e.g. if you ran section 4 of the comparison notebook in this session), then evaluates the **fine-tuned checkpoint** through the identical path (HF pipeline → `predict_depth_meters` → clip → same 7 metrics) and regenerates the combined CSV + charts, now including the `da2-metric-small-ft` row.

In [ ]:
import compare_depth_models as cdm
import finetune_depth_model as fdm

# Baseline first (cached result is reused if present)
cdm.run_comparison(
    models=[cdm.registry_entry(BASELINE_SLUG)],
    num_samples=NUM_EVAL_SAMPLES,
    split="validation",
    output_dir=OUTPUT_DIR,
    device=DEVICE,
    max_depth=10.0,
)

# Fine-tuned checkpoint, identical protocol
ft_result = fdm.evaluate_finetuned(
    FT_CKPT_DIR,
    slug=FT_SLUG,
    name=FT_NAME,
    num_samples=NUM_EVAL_SAMPLES,
    split="validation",
    output_dir=OUTPUT_DIR,
    device=DEVICE,
    max_depth=10.0,
)

# Combined outputs — the fine-tuned row joins every chart
all_results = cdm.load_all_results(OUTPUT_DIR)
cdm.write_metrics_csv(all_results, OUTPUT_DIR)
cdm.plot_comparison_bar_chart(all_results, OUTPUT_DIR)
cdm.plot_per_sample_distribution(all_results, OUTPUT_DIR)
print("\nDone. Outputs in:", OUTPUT_DIR)


## 7. Before → after

In [ ]:
import json
import os
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.width", 160)

def load(slug):
    with open(f"{OUTPUT_DIR}/results/{slug}.json") as f:
        return json.load(f)

base, ft = load(BASELINE_SLUG), load(FT_SLUG)
assert base["status"] == "ok" and ft["status"] == "ok", "baseline or fine-tuned result failed"

rows = []
for field, label, lower_better in cdm.METRIC_SPECS:
    b, f = base["average"][field], ft["average"][field]
    rows.append({
        "metric": label,
        "pretrained": round(b, 4),
        "fine-tuned": round(f, 4),
        "delta": round(f - b, 4),
        "improved": (f < b) if lower_better else (f > b),
    })
display(pd.DataFrame(rows))

# Paired per-sample comparison on AbsRel (same 100 images for both models)
b_ps = [s["abs_rel"] for s in base["per_sample"]]
f_ps = [s["abs_rel"] for s in ft["per_sample"]]
improved = sum(f < b for b, f in zip(b_ps, f_ps))
worsened = sum(f > b for b, f in zip(b_ps, f_ps))
print(f"\nper-image AbsRel: improved on {improved}/{len(b_ps)} images, "
      f"worsened on {worsened}, unchanged on {len(b_ps) - improved - worsened}")

for fig in ["comparison_bar_chart.png", "per_sample_distribution.png"]:
    path = f"{OUTPUT_DIR}/{fig}"
    if os.path.exists(path):
        display(Image(path))


## Notes

- **Comparability:** the fine-tuned result uses the same samples, metrics, clipping and result format as `tsk3succeed.ipynb` — the `da2-metric-small-ft` row can be copied (download `results/da2-metric-small-ft.json`) into the comparison notebook's `OUTPUT_DIR` to appear in its charts there too.
- **Head-only vs full fine-tuning:** with `FREEZE_ENCODER = True` only ~30M of the parameters train. If the gain is small, re-run with `False` (encoder at `LR_ENCODER = 1e-6`) — slower and needs a smaller batch, but usually improves further.
- **Data hygiene:** checkpoint selection uses a held-out slice of the *train* stream; the *validation* split is only touched by the final evaluation. Never train on the validation split — every number in the comparison notebook comes from it.
- **Next models in the series:** duplicate this notebook and change `MODEL_ID` to `...-Base-hf` / `...-Large-hf` (update `FT_CKPT_DIR`, `FT_SLUG`, `FT_NAME`, and `BASELINE_SLUG` accordingly; the Base/Large variants need a smaller `BATCH_SIZE` on a T4).
- Download `FT_CKPT_DIR` (config + weights + `history.json` + `training_curves.png`) and `OUTPUT_DIR/results/da2-metric-small-ft.json` from the Output panel before the session ends.
